# AI-Based Fake News Detection — Training Notebook

This notebook is **Google Colab compatible**. It reproduces the exact same
training pipeline as `src/train.py` in the main project, so you can train
the model in the cloud (with no local setup) and download the resulting
`fake_news_model.pkl` and `tfidf_vectorizer.pkl` files.

**Steps in this notebook:**
1. Install dependencies
2. Download the dataset
3. Explore the dataset
4. Clean the data
5. Preprocess the text (NLP cleaning)
6. Split into train/test sets
7. Vectorize with TF-IDF
8. Train and compare two models (Naive Bayes vs Logistic Regression)
9. Evaluate on the test set
10. Save the trained model + vectorizer (download them to use in the Streamlit app)

In [ ]:
# 1. Install dependencies (Colab already has most of these, this just ensures versions)
!pip install -q scikit-learn pandas numpy nltk joblib

In [ ]:
import nltk
# No NLTK corpus downloads are required; preprocessing uses deterministic Porter stemming.

In [ ]:
# 2. Download the dataset
# This is the same publicly available "Fake or Real News" dataset used by
# the main project (see README.md for full dataset details/source).
!wget -q -O fake_or_real_news.csv.zip "https://github.com/joolsa/fake_real_news_dataset/raw/master/fake_or_real_news.csv.zip"
!unzip -o -q fake_or_real_news.csv.zip
!ls -la fake_or_real_news.csv

In [ ]:
import pandas as pd

df = pd.read_csv('fake_or_real_news.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# 3. Explore the dataset
print(df['label'].value_counts())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# 4. Clean the data
keep_cols = [c for c in ['title', 'text', 'label'] if c in df.columns]
df = df[keep_cols].dropna(subset=['title', 'text', 'label']).drop_duplicates()

df['label'] = df['label'].str.upper().str.strip()
df = df[df['label'].isin(['FAKE', 'REAL'])]
df['label_num'] = (df['label'] == 'REAL').astype(int)  # REAL=1, FAKE=0
df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).str.strip()

print('Shape after cleaning:', df.shape)

In [ ]:
# 5. Preprocess the text
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer
import re

STOPWORDS = set(ENGLISH_STOP_WORDS)
STEMMER = PorterStemmer()
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+')
HTML_PATTERN = re.compile(r'<.*?>')
TOKEN_PATTERN = re.compile(r'[a-zA-Z]{3,}')

def clean_text(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = URL_PATTERN.sub(' ', text)
    text = HTML_PATTERN.sub(' ', text)
    tokens = TOKEN_PATTERN.findall(text)
    tokens = [STEMMER.stem(t) for t in tokens if t not in STOPWORDS]
    return ' '.join(tokens)

df['clean_content'] = df['content'].apply(clean_text)
df = df[df['clean_content'].str.len() > 0]
print('Shape after preprocessing:', df.shape)


In [ ]:
# 6. Split into train/test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_content'], df['label_num'],
    test_size=0.2, random_state=42, stratify=df['label_num']
)
print('Train size:', len(X_train), '| Test size:', len(X_test))

In [ ]:
# 7. Vectorize with TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print('Vocabulary size:', len(vectorizer.vocabulary_))

In [ ]:
# 8. Train and compare two models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def evaluate(name, model):
    y_pred = model.predict(X_test_vec)
    print(f'--- {name} ---')
    print('Accuracy :', accuracy_score(y_test, y_pred))
    print('Precision:', precision_score(y_test, y_pred))
    print('Recall   :', recall_score(y_test, y_pred))
    print('F1-score :', f1_score(y_test, y_pred))
    print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=['FAKE', 'REAL']))
    return f1_score(y_test, y_pred)

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)
nb_f1 = evaluate('Multinomial Naive Bayes', nb_model)

lr_model = LogisticRegression(
    max_iter=2000,
    C=1.5,
    class_weight='balanced',
    solver='liblinear',
    random_state=42,
)
lr_model.fit(X_train_vec, y_train)
lr_f1 = evaluate('Logistic Regression', lr_model)

best_model, best_name = (lr_model, 'Logistic Regression') if lr_f1 >= nb_f1 else (nb_model, 'Multinomial Naive Bayes')
print('\nBest model selected:', best_name)

In [ ]:
# 9 & 10. Save the trained model + vectorizer
import joblib

joblib.dump(best_model, 'fake_news_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print('Saved fake_news_model.pkl and tfidf_vectorizer.pkl')

# In Colab, download them with:
# from google.colab import files
# files.download('fake_news_model.pkl')
# files.download('tfidf_vectorizer.pkl')
#
# Then place both files inside the project's models/ folder to use them
# with app.py (Streamlit) or src/predict.py.